# Notebook 9: Multi-Armed Bandits - Adaptive A/B Testing

**Balancing Exploration and Exploitation**

## Overview

Traditional A/B testing splits traffic equally between variants (e.g., 33% to each email type). But what if we could **learn as we go** and allocate more traffic to the better-performing option?

**Multi-armed bandit algorithms** do exactly that. They continuously balance:
- **Exploration**: Try different options to learn which is best
- **Exploitation**: Allocate more traffic to what's working

Think of it like choosing between slot machines: initially you try each one to see which pays best, but as you learn, you play the best machine more often.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

try:
    import plotly.graph_objects as go
    import plotly.express as px
    PLOTLY_AVAILABLE = True
except ImportError:
    PLOTLY_AVAILABLE = False

sns.set_style("whitegrid")
np.random.seed(42)
plt.rcParams['figure.figsize'] = (12, 6)

# Create output directory
os.makedirs('../data/outputs/nb09', exist_ok=True)


In [ ]:
# Load Hillstrom data
df = pd.read_csv('../data/outputs/nb01/nb01_hillstrom_clean.csv')

# Create our 3 arms
segments = ['Mens E-Mail', 'Womens E-Mail', 'No E-Mail']
arm_names = {
    'Mens E-Mail': 'Arm 0: Men',
    'Womens E-Mail': 'Arm 1: Women',
    'No E-Mail': 'Arm 2: Control'
}

# We'll use conversion as our metric (binary: 0 or 1)
print(f"Dataset size: {len(df)}")
print(f"\nArm allocation (observed):")
for seg in segments:
    n = len(df[df['segment'] == seg])
    rate = df[df['segment'] == seg]['conversion'].mean()
    print(f"  {arm_names[seg]:20s}: n={n:5d}, conversion rate={rate:.3f}")

## The Explore-Exploit Tradeoff

### Exploration
- Trying different options to gather information
- Costs in the short term (maybe picking the worse option)
- Gains knowledge for better long-term decisions

### Exploitation  
- Using the option you believe is best
- Maximizes short-term rewards
- Risks missing better alternatives

### The Dilemma
```
If you ONLY exploit:   You might miss the best option entirely
If you ONLY explore:   You waste resources on bad options even after finding the best
```

Bandits solve this by gradually shifting from exploration to exploitation, guided by data.

### Slot Machine Analogy

Imagine a casino with 3 slot machines:
- You have limited coins (users/budget)
- Each machine has an unknown payout rate
- Should you:
  - Try each equally? (Fixed A/B test)
  - Try to find the best ASAP? (Bandits)
  - Focus on the best once found? (Pure exploitation)

Bandits say: "Try each a bit, then gradually shift your money to the best one you've found."

### Regret

The cost of not always playing the best machine from the start.
- Higher regret = wasted resources
- Lower regret = efficient allocation

## When to Use Bandits vs Traditional A/B Testing

### Fixed-Sample A/B Testing
**Best for:**
- One-time decisions (launch decision)
- Low cost per test
- Need statistical rigor and pre-planned sample size
- Budget is not super tight

**Example:** "Does our new checkout process convert better?"

### Multi-Armed Bandits
**Best for:**
- Ongoing decisions (continuously optimized emails)
- High cost per impression/user
- Want to maximize reward during the test
- Budget is tight and every user matters

**Example:** "Which email variant should we send to maximize conversions?"

### Tradeoffs
| Aspect | A/B Test | Bandits |
|--------|----------|---------|
| Final decision quality | High (pre-powered) | Good (continuous feedback) |
| Reward during test | Low (fixed split) | High (adapts allocation) |
| Complexity | Low | Medium/High |
| Statistical rigor | High | Depends on algorithm |
| Sample size | Pre-planned | Adaptive |

## Epsilon-Greedy Algorithm

**Simplest bandit algorithm**: Maintain success counts for each arm, and:
- With probability (1-ε): Pick the arm with highest **empirical success rate** (exploit)
- With probability ε: Pick a random arm (explore)

### Key Features
- ε (epsilon) controls exploration rate (typically 0.1 = 10%)
- As you gather more data, empirical estimates get better
- Can decay ε over time: ε_t = ε_0 / sqrt(t)

### Intuition
- Start optimistic, try each option a few times
- Once you have some data, mostly pick the best
- Keep small exploration to detect if things change

In [ ]:
def epsilon_greedy(df, outcome_col, n_arms=3, epsilon=0.1, seed=42):
    """
    Simulate epsilon-greedy bandit algorithm on Hillstrom data.
    
    Treat data as a stream of users arriving sequentially.
    At each step, decide which arm to assign based on current estimates.
    """
    np.random.seed(seed)
    
    # Get outcomes for each arm
    arm_data = []
    for i, seg in enumerate(segments):
        outcomes = df[df['segment'] == seg][outcome_col].values
        arm_data.append(outcomes)
    
    # Track results
    n_users = len(df)
    arm_counts = [0] * n_arms  # How many assigned to each arm
    arm_successes = [0] * n_arms  # How many conversions in each arm
    arm_assignment = []  # Which arm assigned to each user
    cumulative_reward = 0
    cumulative_rewards = []
    arm_selection_history = []
    
    # Simulate sequential assignment
    for t in range(n_users):
        # Epsilon-greedy decision
        if np.random.random() < epsilon:
            # Explore: random arm
            chosen_arm = np.random.randint(0, n_arms)
        else:
            # Exploit: best arm (by empirical rate)
            if min(arm_counts) == 0:
                # Not all arms tried yet, try untried
                chosen_arm = np.argmin(arm_counts)
            else:
                # All tried, pick best
                success_rates = [arm_successes[i] / arm_counts[i] if arm_counts[i] > 0 else 0 
                                 for i in range(n_arms)]
                chosen_arm = np.argmax(success_rates)
        
        # Simulate outcome (draw from that arm's data)
        idx_in_arm = arm_counts[chosen_arm] % len(arm_data[chosen_arm])
        outcome = arm_data[chosen_arm][idx_in_arm]
        
        # Update counters
        arm_counts[chosen_arm] += 1
        if outcome == 1:
            arm_successes[chosen_arm] += 1
            cumulative_reward += 1
        
        arm_assignment.append(chosen_arm)
        cumulative_rewards.append(cumulative_reward)
        arm_selection_history.append(chosen_arm)
    
    return {
        'arm_counts': arm_counts,
        'arm_successes': arm_successes,
        'cumulative_rewards': cumulative_rewards,
        'arm_selection_history': arm_selection_history,
        'arm_data': arm_data
    }

# Run epsilon-greedy
eg_results = epsilon_greedy(df, 'conversion', epsilon=0.1)

print("=== Epsilon-Greedy Results ===")
print(f"\nArm Allocation:")
for i, (count, success) in enumerate(zip(eg_results['arm_counts'], eg_results['arm_successes'])):
    rate = success / count if count > 0 else 0
    print(f"  {arm_names[segments[i]]:20s}: {count:5d} assigned, {success:4d} conversions ({rate:.3f})")

print(f"\nTotal conversions (Epsilon-Greedy): {sum(eg_results['arm_successes'])}")
print(f"Final empirical success rates:")
for i, (count, success) in enumerate(zip(eg_results['arm_counts'], eg_results['arm_successes'])):
    rate = success / count if count > 0 else 0
    print(f"  Arm {i}: {rate:.4f}")

In [ ]:
# Visualize epsilon-greedy convergence
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Cumulative rewards over time
axes[0, 0].plot(eg_results['cumulative_rewards'], linewidth=2, color='#1f77b4')
axes[0, 0].set_xlabel('User #', fontsize=11)
axes[0, 0].set_ylabel('Cumulative Conversions', fontsize=11)
axes[0, 0].set_title('Epsilon-Greedy: Cumulative Reward Over Time', fontsize=12, fontweight='bold')
axes[0, 0].grid(alpha=0.3)

# Plot 2: Arm allocation over time (cumulative)
arm_allocation_cumulative = [[], [], []]
counts = [0, 0, 0]
for arm in eg_results['arm_selection_history']:
    counts[arm] += 1
    for i in range(3):
        arm_allocation_cumulative[i].append(counts[i])

for i in range(3):
    axes[0, 1].plot(arm_allocation_cumulative[i], label=f"Arm {i}", linewidth=2)
axes[0, 1].set_xlabel('User #', fontsize=11)
axes[0, 1].set_ylabel('Cumulative Assignments', fontsize=11)
axes[0, 1].set_title('Arm Allocation Over Time', fontsize=12, fontweight='bold')
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

# Plot 3: Final arm allocation (pie chart)
colors_pie = ['#1f77b4', '#ff7f0e', '#2ca02c']
axes[1, 0].pie(eg_results['arm_counts'], labels=[f"Arm {i}
({arm_names[segments[i]].split(':')[1].strip()})" 
                                                   for i in range(3)],
              autopct='%1.1f%%', colors=colors_pie, startangle=90)
axes[1, 0].set_title('Final Arm Allocation (Epsilon-Greedy)', fontsize=12, fontweight='bold')

# Plot 4: Success rate convergence by arm
window_size = 100
arm_rates = [[], [], []]
for t in range(window_size, len(eg_results['arm_selection_history'])):
    recent = eg_results['arm_selection_history'][t-window_size:t]
    for i in range(3):
        recent_arm = [eg_results['arm_selection_history'][j] == i for j in range(t-window_size, t)]
        if sum(recent_arm) > 0:
            recent_outcomes = [eg_results['arm_data'][i][j % len(eg_results['arm_data'][i])] 
                             for j, arm in enumerate(recent_arm) if arm]
            rate = np.mean(recent_outcomes)
        else:
            rate = 0
        arm_rates[i].append(rate)

for i in range(3):
    if arm_rates[i]:
        axes[1, 1].plot(arm_rates[i], label=f"Arm {i}", linewidth=2)
axes[1, 1].set_xlabel('Window #', fontsize=11)
axes[1, 1].set_ylabel('Rolling Success Rate', fontsize=11)
axes[1, 1].set_title(f'Success Rate Convergence (Window={window_size})', fontsize=12, fontweight='bold')
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../data/outputs/nb09/nb09_bandit_epsilon_greedy.png', dpi=150, bbox_inches='tight')
plt.show()

## Thompson Sampling

**Thompson Sampling** uses a Bayesian approach:
1. Maintain a **Beta distribution** for each arm (represents uncertainty about success rate)
2. At each step: Sample from each arm's Beta distribution, pick the highest sample
3. Update Beta distribution when you see an outcome

### Beta Distribution
- Parameterized by α (successes) and β (failures)
- Beta(α, β) represents a distribution of success probabilities
- High α, low β = confident in high success rate
- Equal α, β = maximum uncertainty

### Update Rule (Bayesian)
```
Observe outcome → Update that arm's Beta distribution
Success: α += 1
Failure: β += 1
```

### Why It Works
- Initially, all arms have uncertain (flat) Beta distributions
- Sample from each, picking the highest = explore optimistically
- As evidence accumulates, Beta becomes more concentrated
- Naturally allocates more to arms showing better results

### Intuition
Think of each arm as having a "true" success rate drawn from a Beta distribution. By sampling from our belief about each arm and picking the highest, we naturally emphasize exploring arms that might be best.

In [ ]:
def thompson_sampling(df, outcome_col, n_arms=3, seed=42):
    """Simulate Thompson Sampling bandit algorithm."""
    np.random.seed(seed)
    
    # Get outcomes for each arm
    arm_data = []
    for seg in segments:
        outcomes = df[df['segment'] == seg][outcome_col].values
        arm_data.append(outcomes)
    
    # Beta distribution parameters for each arm: Beta(alpha, beta)
    arm_alpha = [1] * n_arms  # Prior: 1 success
    arm_beta = [1] * n_arms   # Prior: 1 failure
    
    # Track results
    n_users = len(df)
    arm_counts = [0] * n_arms
    arm_successes = [0] * n_arms
    cumulative_reward = 0
    cumulative_rewards = []
    arm_selection_history = []
    beta_evolution = [[] for _ in range(n_arms)]  # Track Beta parameters over time
    
    # Simulate sequential assignment
    for t in range(n_users):
        # Thompson Sampling: sample from each arm's Beta, pick max
        samples = [np.random.beta(arm_alpha[i], arm_beta[i]) for i in range(n_arms)]
        chosen_arm = np.argmax(samples)
        
        # Record posterior at this timestep
        for i in range(n_arms):
            beta_evolution[i].append((arm_alpha[i], arm_beta[i]))
        
        # Simulate outcome
        idx_in_arm = arm_counts[chosen_arm] % len(arm_data[chosen_arm])
        outcome = arm_data[chosen_arm][idx_in_arm]
        
        # Update
        arm_counts[chosen_arm] += 1
        if outcome == 1:
            arm_successes[chosen_arm] += 1
            arm_alpha[chosen_arm] += 1
            cumulative_reward += 1
        else:
            arm_beta[chosen_arm] += 1
        
        arm_selection_history.append(chosen_arm)
        cumulative_rewards.append(cumulative_reward)
    
    return {
        'arm_counts': arm_counts,
        'arm_successes': arm_successes,
        'arm_alpha_final': arm_alpha,
        'arm_beta_final': arm_beta,
        'cumulative_rewards': cumulative_rewards,
        'arm_selection_history': arm_selection_history,
        'beta_evolution': beta_evolution,
        'arm_data': arm_data
    }

ts_results = thompson_sampling(df, 'conversion')

print("=== Thompson Sampling Results ===")
print(f"\nArm Allocation:")
for i, (count, success) in enumerate(zip(ts_results['arm_counts'], ts_results['arm_successes'])):
    rate = success / count if count > 0 else 0
    print(f"  {arm_names[segments[i]]:20s}: {count:5d} assigned, {success:4d} conversions ({rate:.3f})")

print(f"\nFinal Beta Parameters (Alpha, Beta):")
for i in range(3):
    print(f"  Arm {i}: Beta({ts_results['arm_alpha_final'][i]}, {ts_results['arm_beta_final'][i]})")

print(f"\nTotal conversions (Thompson Sampling): {sum(ts_results['arm_successes'])}")

In [ ]:
# Visualize Thompson Sampling - Beta evolution
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Cumulative rewards
axes[0, 0].plot(ts_results['cumulative_rewards'], linewidth=2, color='#ff7f0e')
axes[0, 0].set_xlabel('User #', fontsize=11)
axes[0, 0].set_ylabel('Cumulative Conversions', fontsize=11)
axes[0, 0].set_title('Thompson Sampling: Cumulative Reward', fontsize=12, fontweight='bold')
axes[0, 0].grid(alpha=0.3)

# Plot 2: Arm allocation
arm_allocation_ts = [[], [], []]
counts_ts = [0, 0, 0]
for arm in ts_results['arm_selection_history']:
    counts_ts[arm] += 1
    for i in range(3):
        arm_allocation_ts[i].append(counts_ts[i])

for i in range(3):
    axes[0, 1].plot(arm_allocation_ts[i], label=f"Arm {i}", linewidth=2)
axes[0, 1].set_xlabel('User #', fontsize=11)
axes[0, 1].set_ylabel('Cumulative Assignments', fontsize=11)
axes[0, 1].set_title('Arm Allocation Over Time', fontsize=12, fontweight='bold')
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

# Plot 3: Final arm allocation
colors_pie = ['#1f77b4', '#ff7f0e', '#2ca02c']
axes[1, 0].pie(ts_results['arm_counts'], labels=[f"Arm {i}" for i in range(3)],
              autopct='%1.1f%%', colors=colors_pie, startangle=90)
axes[1, 0].set_title('Final Arm Allocation (Thompson)', fontsize=12, fontweight='bold')

# Plot 4: Beta distribution posteriors at key timepoints
from scipy.stats import beta as beta_dist

x = np.linspace(0, 1, 100)
timepoints = [100, 1000, len(df) // 2, len(df) - 1]

for idx, t in enumerate(timepoints):
    if t < len(ts_results['beta_evolution'][0]):
        alpha_0, beta_0 = ts_results['beta_evolution'][0][t]
        y = beta_dist.pdf(x, alpha_0, beta_0)
        axes[1, 1].plot(x, y, label=f't={t}', linewidth=2)

axes[1, 1].set_xlabel('Success Probability', fontsize=11)
axes[1, 1].set_ylabel('Density', fontsize=11)
axes[1, 1].set_title('Beta Distribution Evolution for Arm 0', fontsize=12, fontweight='bold')
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../data/outputs/nb09/nb09_bandit_thompson_sampling.png', dpi=150, bbox_inches='tight')
plt.show()

## Upper Confidence Bound (UCB1)

**Optimism in the Face of Uncertainty**: Pick the arm with the highest upper confidence bound.

### Algorithm
For each arm:
```
UCB_i(t) = mean_i + sqrt(ln(t) / n_i)
```

Where:
- mean_i = empirical success rate for arm i
- n_i = number of times arm i was chosen
- t = current timestep

The sqrt(ln(t) / n_i) term is the **confidence radius**:
- Small for well-explored arms (large n_i)
- Large for unexplored arms (small n_i)
- Decays over time (ln(t))

### Intuition
"Pick the arm with the best possible success rate (upper bound), considering both empirical mean and uncertainty." Gradually, well-explored arms with low uncertainty dominate.

In [ ]:
def ucb1_bandit(df, outcome_col, n_arms=3, seed=42):
    """Simulate UCB1 bandit algorithm."""
    np.random.seed(seed)
    
    arm_data = []
    for seg in segments:
        outcomes = df[df['segment'] == seg][outcome_col].values
        arm_data.append(outcomes)
    
    n_users = len(df)
    arm_counts = [0] * n_arms
    arm_successes = [0] * n_arms
    cumulative_reward = 0
    cumulative_rewards = []
    arm_selection_history = []
    
    for t in range(1, n_users):
        # Calculate UCB for each arm
        ucb_values = []
        for i in range(n_arms):
            if arm_counts[i] == 0:
                ucb = float('inf')  # Unvisited arms
            else:
                mean = arm_successes[i] / arm_counts[i]
                confidence_radius = np.sqrt(np.log(t) / arm_counts[i])
                ucb = mean + confidence_radius
            ucb_values.append(ucb)
        
        # Pick arm with highest UCB
        chosen_arm = np.argmax(ucb_values)
        
        # Simulate outcome
        idx_in_arm = arm_counts[chosen_arm] % len(arm_data[chosen_arm])
        outcome = arm_data[chosen_arm][idx_in_arm]
        
        # Update
        arm_counts[chosen_arm] += 1
        if outcome == 1:
            arm_successes[chosen_arm] += 1
            cumulative_reward += 1
        
        arm_selection_history.append(chosen_arm)
        cumulative_rewards.append(cumulative_reward)
    
    return {
        'arm_counts': arm_counts,
        'arm_successes': arm_successes,
        'cumulative_rewards': cumulative_rewards,
        'arm_selection_history': arm_selection_history,
        'arm_data': arm_data
    }

ucb_results = ucb1_bandit(df, 'conversion')

print("=== UCB1 Results ===")
print(f"\nArm Allocation:")
for i, (count, success) in enumerate(zip(ucb_results['arm_counts'], ucb_results['arm_successes'])):
    rate = success / count if count > 0 else 0
    print(f"  {arm_names[segments[i]]:20s}: {count:5d} assigned, {success:4d} conversions ({rate:.3f})")

print(f"\nTotal conversions (UCB1): {sum(ucb_results['arm_successes'])}")

In [ ]:
# Comprehensive comparison of all three algorithms
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Cumulative rewards comparison
axes[0, 0].plot(eg_results['cumulative_rewards'], label='Epsilon-Greedy', linewidth=2.5)
axes[0, 0].plot(ts_results['cumulative_rewards'], label='Thompson Sampling', linewidth=2.5)
axes[0, 0].plot(ucb_results['cumulative_rewards'], label='UCB1', linewidth=2.5)
axes[0, 0].set_xlabel('User #', fontsize=11)
axes[0, 0].set_ylabel('Cumulative Conversions', fontsize=11)
axes[0, 0].set_title('Cumulative Reward Comparison', fontsize=12, fontweight='bold')
axes[0, 0].legend(fontsize=10)
axes[0, 0].grid(alpha=0.3)

# Final arm allocation comparison
algorithms = ['E-Greedy', 'Thompson', 'UCB1']
x_pos = np.arange(3)
width = 0.25

for idx, arm_idx in enumerate([0, 1, 2]):
    allocations = [
        eg_results['arm_counts'][arm_idx] / len(df) * 100,
        ts_results['arm_counts'][arm_idx] / len(df) * 100,
        ucb_results['arm_counts'][arm_idx] / len(df) * 100
    ]
    axes[0, 1].bar(x_pos + idx*width, allocations, width, label=f'Arm {arm_idx}')

axes[0, 1].set_xlabel('Algorithm', fontsize=11)
axes[0, 1].set_ylabel('Allocation (%)', fontsize=11)
axes[0, 1].set_title('Final Arm Allocation by Algorithm', fontsize=12, fontweight='bold')
axes[0, 1].set_xticks(x_pos + width)
axes[0, 1].set_xticklabels(algorithms)
axes[0, 1].legend()
axes[0, 1].grid(axis='y', alpha=0.3)

# Regret comparison (conversions lost vs best possible)
best_arm = np.argmax([arm_data[i].mean() for i, arm_data in enumerate(eg_results['arm_data'])])
best_rate = eg_results['arm_data'][best_arm].mean()

eg_regrets = [best_rate * (i+1) - r for i, r in enumerate(eg_results['cumulative_rewards'])]
ts_regrets = [best_rate * (i+1) - r for i, r in enumerate(ts_results['cumulative_rewards'])]
ucb_regrets = [best_rate * (i+1) - r for i, r in enumerate(ucb_results['cumulative_rewards'])]

axes[1, 0].plot(eg_regrets, label='Epsilon-Greedy', linewidth=2.5)
axes[1, 0].plot(ts_regrets, label='Thompson Sampling', linewidth=2.5)
axes[1, 0].plot(ucb_regrets, label='UCB1', linewidth=2.5)
axes[1, 0].set_xlabel('User #', fontsize=11)
axes[1, 0].set_ylabel('Cumulative Regret (missed conversions)', fontsize=11)
axes[1, 0].set_title('Cumulative Regret Comparison', fontsize=12, fontweight='bold')
axes[1, 0].legend(fontsize=10)
axes[1, 0].grid(alpha=0.3)

# Summary statistics
summary_data = {
    'Algorithm': ['Epsilon-Greedy', 'Thompson Sampling', 'UCB1'],
    'Total Conversions': [
        sum(eg_results['arm_successes']),
        sum(ts_results['arm_successes']),
        sum(ucb_results['arm_successes'])
    ],
    'Best Arm Allocation %': [
        eg_results['arm_counts'][best_arm] / len(df) * 100,
        ts_results['arm_counts'][best_arm] / len(df) * 100,
        ucb_results['arm_counts'][best_arm] / len(df) * 100
    ],
    'Final Regret': [
        eg_regrets[-1],
        ts_regrets[-1],
        ucb_regrets[-1]
    ]
}

summary_df = pd.DataFrame(summary_data)

# Plot as table
axes[1, 1].axis('tight')
axes[1, 1].axis('off')
table = axes[1, 1].table(cellText=summary_df.values, colLabels=summary_df.columns,
                         cellLoc='center', loc='center', colWidths=[0.25, 0.2, 0.25, 0.2])
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 2)
axes[1, 1].set_title('Performance Comparison', fontsize=12, fontweight='bold', pad=20)

plt.tight_layout()
plt.savefig('../data/outputs/nb09/nb09_bandit_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n=== Algorithm Comparison ===")
print(summary_df.to_string(index=False))

In [ ]:
# Compare bandit allocation to fixed equal split
fixed_allocation = len(df) // 3  # 33% to each

print("\n=== Bandits vs Traditional A/B Test ===")
print(f"\nFixed 33% Split (Traditional A/B):")
for i, seg in enumerate(segments):
    print(f"  Arm {i}: {fixed_allocation} users")
print(f"  Total conversions: {fixed_allocation * best_rate * 3:.0f}")

print(f"\nBandit Allocations:")
algorithms = ['Epsilon-Greedy', 'Thompson Sampling', 'UCB1']
bandit_results = [eg_results, ts_results, ucb_results]

for algo_name, results in zip(algorithms, bandit_results):
    total_conv = sum(results['arm_successes'])
    efficiency_gain = ((total_conv - (fixed_allocation * best_rate * 3)) / 
                      (fixed_allocation * best_rate * 3) * 100)
    print(f"\n{algo_name}:")
    print(f"  Total conversions: {total_conv}")
    print(f"  Efficiency gain vs fixed: {efficiency_gain:.1f}%")
    print(f"  Allocation: ", end="")
    for count in results['arm_counts']:
        print(f"{count:5d} ", end="")
    print()

## Contextual Bandits (Brief Introduction)

All algorithms above treat customers as identical and only learn the arm's global success rate.

**Contextual Bandits** extend this by using customer features:

### Example
Instead of:
- "Email is 30% conversion, Control is 25%"

You learn:
- "Email is 35% conversion for high-value customers, 25% for low-value"
- "Control is 15% for high-value, 30% for low-value"

### Algorithms
- **LinUCB**: Linear regression + UCB
- **Thompson Sampling with features**: Bayesian linear regression per arm
- **ε-greedy with clustering**: Cluster customers, run separate bandits

### Tradeoff
- Better targeting
- But need more data to learn per context
- Complexity increases

In [ ]:
# Save bandit simulation results
bandit_results_df = pd.DataFrame({
    'Algorithm': ['Epsilon-Greedy'] * 3 + ['Thompson Sampling'] * 3 + ['UCB1'] * 3,
    'Arm': [0, 1, 2] * 3,
    'Users_Assigned': (eg_results['arm_counts'] + ts_results['arm_counts'] + ucb_results['arm_counts']),
    'Conversions': (eg_results['arm_successes'] + ts_results['arm_successes'] + ucb_results['arm_successes']),
    'Conversion_Rate': [
        eg_results['arm_successes'][i] / eg_results['arm_counts'][i] if eg_results['arm_counts'][i] > 0 else 0
        for i in range(3)
    ] + [
        ts_results['arm_successes'][i] / ts_results['arm_counts'][i] if ts_results['arm_counts'][i] > 0 else 0
        for i in range(3)
    ] + [
        ucb_results['arm_successes'][i] / ucb_results['arm_counts'][i] if ucb_results['arm_counts'][i] > 0 else 0
        for i in range(3)
    ]
})

bandit_results_df.to_csv('../data/outputs/nb09/nb09_bandit_results.csv', index=False)
print("Bandit results saved to: ../data/outputs/nb09/nb09_bandit_results.csv")
print("\nResults:")
print(bandit_results_df.to_string(index=False))

## Key Takeaways

1. **Three Main Algorithms**:
   - **Epsilon-Greedy**: Simple, easy to implement, good baseline
   - **Thompson Sampling**: Elegant Bayesian approach, very effective
   - **UCB1**: Optimistic exploration, strong theoretical guarantees

2. **Bandits Win When**:
   - Cost per user is high (focus on best option ASAP)
   - Experiment is ongoing (continuous optimization)
   - Reward during experiment matters
   
3. **Traditional A/B Test Wins When**:
   - Need pre-planned, statistically rigorous decisions
   - One-time decision
   - Cost per user is low

4. **Practical Efficiency**: Bandits typically achieve 10-30% higher cumulative reward vs fixed allocation during the experiment

5. **Personalization**: Contextual bandits extend basic algorithms to account for customer features

## When to Use Each Algorithm

- **Epsilon-Greedy**: Getting started, simple implementation
- **Thompson Sampling**: Production systems, best balance of simplicity and performance
- **UCB1**: When you want theoretical guarantees
- **Contextual**: When personalization is critical